# Lab 21-03 · Noise Figure, Noise Parameters and Intermodulation

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import cmath
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import rfnl as r

P = lambda m, d: m * cmath.exp(1j * math.radians(d))


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}


FS, N, F1, F2 = 64e6, 6400, 9e6, 10e6


def line(f, sp, f0):
    return sp[np.argmin(abs(f - f0))]

## Part 1 · Noise over the source plane

*Predict* where the minimum lies and whether 50 Ω or the power-match point is closer to it.

In [2]:
gopt, gconj = P(0.45, 60), P(0.6, -40)
x = np.linspace(-0.99, 0.99, 301)
gx, gy = np.meshgrid(x, x)
gs = gx + 1j * gy
inside = abs(gs) < 0.99
nf = np.full(gs.shape, np.nan)
nf[inside] = 10 * np.log10(r.noise_factor_gamma(gs[inside], 0.8, 8.0, gopt))
plt.figure(figsize=(5, 5))
cs = plt.contour(gx, gy, nf, levels=[0.9, 1.0, 1.5, 2.0, 3.0])
plt.clabel(cs, fmt="%.1f dB")
for pt, lab in ((gopt, "opt"), (0, "50 ohm"), (gconj, "power match")):
    plt.plot([np.real(pt)], [np.imag(pt)], "o"); plt.annotate(lab, (np.real(pt), np.imag(pt)))
plt.gca().set_aspect("equal"); plt.show()
for pt in (gopt, 0, gconj):
    print(f"Gamma_s = {abs(pt):.2f} at {math.degrees(cmath.phase(pt)) if pt else 0:.0f} deg: "
          f"NF = {10 * math.log10(r.noise_factor_gamma(pt, 0.8, 8.0, gopt)):.2f} dB")
for nfi in (1.0, 1.5):
    c, rad = r.noise_circle(nfi, 0.8, 8.0, gopt)
    print(f"{nfi} dB circle: centre {abs(c):.3f}, radius {rad:.3f}")

Gamma_s = 0.45 at 60 deg: NF = 0.80 dB
Gamma_s = 0.00 at 0 deg: NF = 1.07 dB
Gamma_s = 0.60 at -40 deg: NF = 2.04 dB
1.0 dB circle: centre 0.393, radius 0.324
1.5 dB circle: centre 0.292, radius 0.553


## Part 2 · Y-factor statistics

*Predict* the scatter of the measured noise figure when each noise power is averaged over $10^4$ samples.

In [3]:
rng = np.random.default_rng(1)
T0 = 290.0
te12 = (10 ** (r.y_factor_nf(15.0, 13.5) / 10) - 1) * T0
t_on, t_off = T0 * (10 ** 1.5 + 1) + te12, T0 + te12
nfs = []
for _ in range(2000):
    pon = np.mean(rng.normal(0, math.sqrt(t_on), 10_000) ** 2)
    poff = np.mean(rng.normal(0, math.sqrt(t_off), 10_000) ** 2)
    nfs.append(10 * math.log10(10 ** 1.5 / (pon / poff - 1)))
y = 10 ** 1.35
print(f"mean NF {np.mean(nfs):.3f} dB, std {np.std(nfs):.3f} dB; predicted std "
      f"{10 / math.log(10) * y / (y - 1) * 2 / 100:.3f} dB")
plt.figure(); plt.hist(nfs, bins=40); plt.xlabel("measured NF (dB)"); plt.show()

mean NF 1.700 dB, std 0.091 dB; predicted std 0.091 dB


## Part 3 · Two-tone sweep

*Predict* the IM3 slope at small inputs and what happens near compression.

In [4]:
a1 = 10 ** (18 / 20)
a3 = -4 / 3 * a1 / r.amp_from_dbm(0.0) ** 2
amp = lambda x: a1 * x + a3 * x ** 3
pins = np.arange(-60, 1, 2.0)
fo, im = [], []
for p in pins:
    a = r.amp_from_dbm(p)
    f, sp = r.tones(amp, [a, a], [F1, F2], FS, N)
    fo.append(r.dbm_from_amp(line(f, sp, F1))); im.append(r.dbm_from_amp(line(f, sp, 2 * F1 - F2)))
fo, im = np.array(fo), np.array(im)
slope = (im[pins == -40][0] - im[pins == -50][0]) / 10
print(f"IM3 slope between -50 and -40 dBm: {slope:.2f}; IIP3 from -30 dBm: "
      f"{-30 + (fo[pins == -30][0] - im[pins == -30][0]) / 2:.2f} dBm")
plt.figure(); plt.plot(pins, fo, label="fundamental"); plt.plot(pins, im, label="IM3")
plt.plot(pins, pins + 18, "--", lw=0.8); plt.plot(pins, 3 * pins + 18, "--", lw=0.8)
plt.ylim(-140, 30); plt.xlabel("input per tone (dBm)"); plt.ylabel("output (dBm)"); plt.legend(); plt.show()

IM3 slope between -50 and -40 dBm: 3.00; IIP3 from -30 dBm: -0.01 dBm


## Part 4 · Transistors, exactly and in ngspice

*Predict* HD2, HD3 and IM3 for a bipolar transistor at 5 mV, and IM3 for a differential pair at 10 mV per tone.

In [5]:
vt = r.K_B * 300 / r.Q_E
x = 0.005 / vt
hd2, hd3 = r.bjt_harmonics(x)
f, sp = r.tones(lambda u: np.exp(u / vt), [0.005, 0.005], [F1, F2], FS, N)
im3 = 20 * math.log10(line(f, sp, 2 * F1 - F2) / line(f, sp, F1))
f, sp = r.tones(lambda u: np.tanh(u / (2 * vt)), [0.01, 0.01], [F1, F2], FS, N)
im3dp = 20 * math.log10(line(f, sp, 2 * F1 - F2) / line(f, sp, F1))
print(f"Python : HD2 {hd2 * 100:.3f} %, HD3 {hd3 * 100:.4f} %, IM3 {im3:.2f} dBc, pair IM3 {im3dp:.2f} dBc")
s = ngspice("bjt_im.cir")
print(f"ngspice: HD2 {s['hd2'] * 100:.3f} %, HD3 {s['hd3'] * 100:.4f} %, IM3 {s['im3']:.2f} dBc, "
      f"pair IM3 {s['im3dp']:.2f} dBc, pair IM2 residue {s['im2dp']:.1e}")

Python : HD2 4.828 %, HD3 0.1555 %, IM3 -46.66 dBc, pair IM3 -40.87 dBc
ngspice: HD2 4.828 %, HD3 0.1555 %, IM3 -46.66 dBc, pair IM3 -40.87 dBc, pair IM2 residue 1.2e-08


## Part 5 · The line-up

*Predict* how the cascade noise figure and IIP3 move as the LNA gain rises.

In [6]:
gains = np.arange(5, 31, 1.0)
nfs = [r.friis([(-2, 2), (g, 1.2), (8, 10), (20, 15)]) for g in gains]
ips = [r.cascade_iip3([(-2, 40), (g, -8), (8, 5), (20, 15)]) for g in gains]
fig, ax = plt.subplots(); ax.plot(gains, nfs, label="NF (dB)"); ax.set_xlabel("LNA gain (dB)"); ax.set_ylabel("NF (dB)")
ax2 = ax.twinx(); ax2.plot(gains, ips, "C1", label="IIP3 (dBm)"); ax2.set_ylabel("IIP3 (dBm)"); plt.show()
for g in (10, 15, 20, 25):
    print(f"LNA gain {g} dB: NF {r.friis([(-2, 2), (g, 1.2), (8, 10), (20, 15)]):.2f} dB, "
          f"IIP3 {r.cascade_iip3([(-2, 40), (g, -8), (8, 5), (20, 15)]):.1f} dBm")

LNA gain 10 dB: NF 6.32 dB, IIP3 -8.6 dBm
LNA gain 15 dB: NF 4.45 dB, IIP3 -11.5 dBm
LNA gain 20 dB: NF 3.63 dB, IIP3 -15.6 dBm
LNA gain 25 dB: NF 3.34 dB, IIP3 -20.3 dBm
